# 🤖 Resume Generator — AI-Tailored Resumes per User per Job

**Purpose**: For each pending `user_job_matches` record:
- Fetch user's resume (work history, skills, education)
- Fetch job's JD, roles, requirements
- Call NVIDIA NIM AI to tailor bullets + summary
- Build professional DOCX
- Save DOCX + PDF paths to `generated_resumes`
- Create email draft in `email_drafts`

**Rules**:
- Different bullets for every user-job combo (never same resume twice)
- Company names kept real; only bullet content is tailored
- `basic_knowledge_skills` from clipboard → added as 'Familiar with X'
- 10 resumes per user per run (default; UI can trigger more)

In [0]:
import os, re, json, uuid, requests, time, hashlib, base64
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

spark = SparkSession.builder.getOrCreate()

CATALOG         = 'jobs_automation_db'
MATCHES_TABLE   = f'{CATALOG}.default.user_job_matches'
SILVER_TABLE    = f'{CATALOG}.default.jobs_clean_silver'
USERS_TABLE     = f'{CATALOG}.users_schema.users'
RESUMES_TABLE   = f'{CATALOG}.users_schema.user_resumes'
CLIPBOARD_TABLE = f'{CATALOG}.users_schema.user_application_clipboard'
GEN_RESUMES_TBL = f'{CATALOG}.default.generated_resumes'
EMAIL_DRAFTS_TBL= f'{CATALOG}.default.email_drafts'

NVIDIA_API_KEY  = dbutils.secrets.get('jobs_automation', 'nvidia_nim')
NVIDIA_MODEL    = 'meta/llama-3.1-8b-instruct'
NVIDIA_URL      = 'https://integrate.api.nvidia.com/v1/chat/completions'

RESUMES_PER_USER  = 10   # Default batch size
PARALLEL_WORKERS  = 5    # Parallel resume generation
DBFS_RESUME_DIR   = '/FileStore/resumes/generated/'
LOCAL_RESUME_DIR  = '/tmp/resumes_gen/'

os.makedirs(LOCAL_RESUME_DIR, exist_ok=True)
print('✅ Setup complete')

In [0]:
# Get top pending matches per user (ordered by match_score DESC)
pending_df = spark.sql(f'''
    WITH ranked AS (
        SELECT m.*,
               ROW_NUMBER() OVER (PARTITION BY m.user_id ORDER BY m.match_score DESC) AS rn
        FROM {MATCHES_TABLE} m
        WHERE m.status = 'matched'
          AND m.resume_generated = false
    )
    SELECT * FROM ranked WHERE rn <= {RESUMES_PER_USER}
''')

pending_matches = pending_df.collect()
print(f'📋 Pending matches to process: {len(pending_matches)}')

if len(pending_matches) == 0:
    print('ℹ️  No pending matches. All resumes up to date!')
    dbutils.notebook.exit('No work to do')

In [0]:
def ai_tailor_resume(user_data: dict, job_data: dict, clipboard: dict) -> dict:
    '''
    Call NVIDIA NIM to tailor resume content.
    Returns dict with: summary, tailored_bullets (per company), skills_to_highlight
    '''
    work_history = json.loads(user_data.get('work_history_json') or '[]')
    user_skills  = user_data.get('skills_extracted', '')
    exp_years    = clipboard.get('years_experience_override') or user_data.get('years_experience', 0)
    extra_skills = clipboard.get('basic_knowledge_skills', '')  # Skills to add as 'Familiar with'

    # Build work history summary for prompt (don't expose company names to AI — just roles + bullets)
    work_summary = ''
    for i, job in enumerate(work_history[-4:]):  # Last 4 jobs max
        title   = job.get('title', 'Engineer')
        bullets = '\n'.join(f'  - {b}' for b in job.get('bullets', [])[:5])
        work_summary += f'\nRole {i+1}: {title}\n{bullets}\n'

    jd_text = (job_data.get('job_description') or '')[:2500]
    roles   = job_data.get('roles_responsibilities') or job_data.get('roles_summary') or ''
    reqs    = job_data.get('requirements_section') or ''
    tech    = job_data.get('tech_stack') or ''

    prompt = f'''You are an expert resume writer for US IT jobs. Tailor this resume for the job below.

=== CANDIDATE BACKGROUND ===
Total Experience: {exp_years} years
Skills: {user_skills}
Work History (roles only, no company names):
{work_summary}

=== TARGET JOB ===
Title: {job_data["job_title"]}
Location: {job_data["location"]}
Required Tech Stack: {tech}
Key Responsibilities: {roles[:800]}
Requirements: {reqs[:800]}
Full JD: {jd_text}

=== INSTRUCTIONS ===
1. Write a 3-sentence professional summary tailored to THIS job. Start with experience years.
2. For EACH of the {len(work_history[-4:])} roles, write 4-5 STAR-format bullet points.
   - Each bullet must use ACTION verbs (Built, Designed, Optimized, Led, Implemented)
   - Include measurable results: percentages, time saved, data volumes
   - Naturally embed keywords from the job's tech stack
   - Make bullets sound like a REAL human wrote them, not AI
   - DO NOT mention company names in bullets
3. List top 8 skills to highlight from the candidate's stack that match this job
4. If extra skills provided, note them as "Familiar with: X, Y"

Extra skills to mention as familiar: {extra_skills}

Respond ONLY with valid JSON:
{{
  "professional_summary": "<3 sentences>",
  "role_bullets": [
    {{"role_index": 0, "bullets": ["<bullet1>", "<bullet2>", "<bullet3>", "<bullet4>"]}},
    {{"role_index": 1, "bullets": ["<bullet1>", "<bullet2>", "<bullet3>", "<bullet4>"]}}
  ],
  "skills_highlighted": ["skill1", "skill2", "skill3", "skill4", "skill5", "skill6", "skill7", "skill8"],
  "familiar_with": "<comma-separated if any, else empty string>"
}}'''

    for attempt in range(3):
        try:
            resp = requests.post(
                NVIDIA_URL,
                headers={'Authorization': f'Bearer {NVIDIA_API_KEY}', 'Content-Type': 'application/json'},
                json={'model': NVIDIA_MODEL,
                      'messages': [{'role': 'user', 'content': prompt}],
                      'temperature': 0.4,  # Slight variation so resumes differ
                      'max_tokens': 1200},
                timeout=45,
            )
            if resp.status_code == 200:
                content = resp.json()['choices'][0]['message']['content'].strip()
                m = re.search(r'\{.*\}', content, re.DOTALL)
                if m:
                    return json.loads(m.group())
            elif resp.status_code == 429:
                time.sleep(15 * (attempt + 1))
        except json.JSONDecodeError:
            pass
        except Exception as e:
            print(f'  AI error attempt {attempt+1}: {e}')
            time.sleep(5)

    # Fallback: use original bullets
    return {
        'professional_summary': f'Experienced {job_data["job_title"]} with {exp_years}+ years in data engineering. Skilled in {tech[:100]}. Proven track record of delivering scalable solutions.',
        'role_bullets': [{'role_index': i, 'bullets': job.get('bullets', [])[:4]} for i, job in enumerate(work_history[-4:])],
        'skills_highlighted': (user_skills or '').split(',')[:8],
        'familiar_with': extra_skills,
    }

print('✅ AI tailor function ready')

In [0]:
def build_resume_docx(user_data: dict, job_data: dict, ai_content: dict,
                      clipboard: dict, output_path: str):
    '''
    Build a professional ATS-friendly DOCX resume.
    - Clean single-column layout (passes ATS parsers)
    - Uses actual company names from work history
    - AI-tailored bullets injected per role
    '''
    doc = Document()

    # ── Page margins (narrow for more content) ──────────────
    for section in doc.sections:
        section.top_margin    = Inches(0.6)
        section.bottom_margin = Inches(0.6)
        section.left_margin   = Inches(0.75)
        section.right_margin  = Inches(0.75)

    # ── Helper: add styled paragraph ────────────────────────
    def add_para(text, bold=False, size=11, color=None, align=WD_ALIGN_PARAGRAPH.LEFT, space_after=2):
        p = doc.add_paragraph()
        p.alignment = align
        p.paragraph_format.space_after = Pt(space_after)
        p.paragraph_format.space_before = Pt(0)
        run = p.add_run(text)
        run.bold = bold
        run.font.size = Pt(size)
        if color:
            run.font.color.rgb = RGBColor(*color)
        return p

    def add_section_header(title):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(6)
        p.paragraph_format.space_after  = Pt(2)
        run = p.add_run(title.upper())
        run.bold = True
        run.font.size = Pt(11)
        run.font.color.rgb = RGBColor(31, 73, 125)  # Professional dark blue
        # Underline via border
        from docx.oxml.ns import qn
        from docx.oxml import OxmlElement
        pPr = p._p.get_or_add_pPr()
        pBdr = OxmlElement('w:pBdr')
        bottom = OxmlElement('w:bottom')
        bottom.set(qn('w:val'), 'single')
        bottom.set(qn('w:sz'), '4')
        bottom.set(qn('w:space'), '1')
        bottom.set(qn('w:color'), '1F497D')
        pBdr.append(bottom)
        pPr.append(pBdr)
        return p

    # Parse user profile
    exp_years   = clipboard.get('years_experience_override') or user_data.get('years_experience', 0)
    phone       = clipboard.get('phone_override') or user_data.get('phone', '')
    location    = clipboard.get('location_override') or f"{user_data.get('city','')}, {user_data.get('state','')}"
    linkedin_url= user_data.get('linkedin_url', '')
    github_url  = user_data.get('github_url', '')
    work_history= json.loads(user_data.get('work_history_json') or '[]')
    education   = json.loads(user_data.get('education_json') or '[]')
    certs       = user_data.get('certifications', '')

    ai_bullets_map = {item['role_index']: item['bullets'] for item in ai_content.get('role_bullets', [])}
    skills_list    = ai_content.get('skills_highlighted', [])
    familiar_with  = ai_content.get('familiar_with', '')

    # ══════════════════════════════════════════
    # HEADER
    # ══════════════════════════════════════════
    name_para = doc.add_paragraph()
    name_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    name_para.paragraph_format.space_after = Pt(2)
    name_run = name_para.add_run(user_data.get('full_name', 'Candidate Name'))
    name_run.bold = True
    name_run.font.size = Pt(16)
    name_run.font.color.rgb = RGBColor(31, 73, 125)

    # Contact line
    contact_parts = [x for x in [phone, user_data.get('email_address',''), location] if x and x.strip(', ')]
    contact_line = '  |  '.join(contact_parts)
    if linkedin_url:
        contact_line += f'  |  {linkedin_url}'
    if github_url:
        contact_line += f'  |  {github_url}'
    add_para(contact_line, size=9, align=WD_ALIGN_PARAGRAPH.CENTER, space_after=4)

    # ══════════════════════════════════════════
    # PROFESSIONAL SUMMARY
    # ══════════════════════════════════════════
    add_section_header('Professional Summary')
    summary = ai_content.get('professional_summary', '')
    add_para(summary, size=10, space_after=4)

    # ══════════════════════════════════════════
    # TECHNICAL SKILLS
    # ══════════════════════════════════════════
    add_section_header('Technical Skills')
    all_skills = skills_list[:12]
    skills_str = '  •  '.join(all_skills)
    add_para(skills_str, size=10, space_after=2)
    if familiar_with:
        add_para(f'Familiar with: {familiar_with}', size=10, space_after=4)

    # ══════════════════════════════════════════
    # PROFESSIONAL EXPERIENCE
    # ══════════════════════════════════════════
    add_section_header('Professional Experience')

    recent_jobs = work_history[-4:]  # Last 4 jobs
    for i, job in enumerate(reversed(recent_jobs)):  # Most recent first
        role_idx = len(recent_jobs) - 1 - i

        company = job.get('company', 'Company Name')
        title   = job.get('title', 'Engineer')
        start   = job.get('start', '')
        end     = job.get('end', 'Present')

        # Job header: Title | Company | Dates
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(4)
        p.paragraph_format.space_after  = Pt(1)
        r1 = p.add_run(f'{title}')
        r1.bold = True
        r1.font.size = Pt(10.5)
        r2 = p.add_run(f'  —  {company}')
        r2.font.size = Pt(10)
        r2.font.color.rgb = RGBColor(89, 89, 89)
        r3 = p.add_run(f'  |  {start} – {end}')
        r3.font.size = Pt(9.5)
        r3.italic = True
        r3.font.color.rgb = RGBColor(120, 120, 120)

        # Bullets — use AI-tailored if available, else original
        bullets = ai_bullets_map.get(role_idx) or job.get('bullets', [])[:5]
        for bullet in bullets:
            bp = doc.add_paragraph(style='List Bullet')
            bp.paragraph_format.space_after = Pt(1)
            bp.paragraph_format.left_indent = Inches(0.2)
            br = bp.add_run(bullet)
            br.font.size = Pt(10)

    # ══════════════════════════════════════════
    # EDUCATION
    # ══════════════════════════════════════════
    if education:
        add_section_header('Education')
        for edu in education:
            p = doc.add_paragraph()
            p.paragraph_format.space_after = Pt(2)
            r1 = p.add_run(f"{edu.get('degree', 'Degree')}")
            r1.bold = True
            r1.font.size = Pt(10)
            r2 = p.add_run(f"  —  {edu.get('school', '')}  |  {edu.get('year', '')}")
            r2.font.size = Pt(10)
            r2.font.color.rgb = RGBColor(89, 89, 89)

    # ══════════════════════════════════════════
    # CERTIFICATIONS
    # ══════════════════════════════════════════
    if certs:
        add_section_header('Certifications')
        for cert in certs.split(','):
            c = cert.strip()
            if c:
                cp = doc.add_paragraph(style='List Bullet')
                cp.paragraph_format.space_after = Pt(1)
                cr = cp.add_run(c)
                cr.font.size = Pt(10)

    doc.save(output_path)
    return output_path

print('✅ DOCX builder ready')

In [0]:
def build_email_draft(user_data: dict, job_data: dict, resume_id: str,
                      match_id: str, email_config: dict) -> dict:
    '''
    Generate a professional cold email draft for a job application.
    If HR email not found → status = 'no_hr_email' (user applies via link)
    '''
    hr_email = job_data.get('hr_email') or ''
    job_title= job_data.get('job_title', '')
    company  = job_data.get('company_name', '')
    tech     = job_data.get('tech_stack', '')
    exp_years= user_data.get('years_experience', 0)
    full_name= user_data.get('full_name', '')

    subject  = f'Application: {job_title} — {full_name} | {exp_years}+ Years Experience'

    skills_preview = ', '.join((tech or '').split(',')[:4])

    body_text = f'''Dear Hiring Team,

I am writing to express my strong interest in the {job_title} position at {company}. With {exp_years}+ years of hands-on experience in {skills_preview}, I am confident in my ability to contribute effectively to your team from day one.

Throughout my career, I have successfully designed and delivered end-to-end data engineering solutions, working extensively with the technologies highlighted in your job description. My background aligns well with your requirements for this role.

Please find my tailored resume attached for your review. I would welcome the opportunity to discuss how my expertise can benefit {company}.

Thank you for your time and consideration.

Best regards,
{full_name}
{user_data.get('phone', '')}
{user_data.get('linkedin_url', '')}
'''

    body_html = body_text.replace('\n', '<br>')

    return {
        'draft_id':           str(uuid.uuid4()),
        'user_id':            user_data.get('user_id'),
        'job_hash':           job_data.get('job_hash'),
        'match_id':           match_id,
        'resume_id':          resume_id,
        'from_email':         email_config.get('email_address', ''),
        'to_email':           hr_email,
        'subject':            subject,
        'body_html':          body_html,
        'body_text':          body_text,
        'draft_type':         'initial_apply',
        'parent_draft_id':    None,
        'recruiter_email_raw':None,
        'status':             'draft' if hr_email else 'no_hr_email',
        'created_at':         datetime.now(),
        'sent_at':            None,
        'smtp_response':      None,
        'is_user_edited':     False,
    }

print('✅ Email draft builder ready')

In [0]:
# Pre-load user data, jobs, resumes into dicts for fast lookup
users_dict   = {r.user_id: r.asDict() for r in spark.sql(f'''SELECT u.*, r.skills_extracted, r.years_experience, r.resume_id, r.work_history_json, r.education_json, r.certifications FROM {USERS_TABLE} u LEFT JOIN {RESUMES_TABLE} r ON u.user_id = r.user_id AND r.is_primary = true''').collect()}
jobs_dict    = {r.job_hash: r.asDict() for r in spark.sql(f'SELECT * FROM {SILVER_TABLE}').collect()}
clips_dict   = {r.user_id: r.asDict() for r in spark.sql(f'SELECT * FROM {CLIPBOARD_TABLE} WHERE is_default = true').collect()}

# Try to get email config per user
try:
    emails_dict = {r.user_id: r.asDict() for r in spark.sql(f'SELECT * FROM {CATALOG}.users_schema.user_emails WHERE is_primary = true').collect()}
except Exception:
    emails_dict = {}

generated_records = []
email_draft_records = []
match_updates = []  # (match_id, resume_id, status)

def process_one_match(match):
    user_id  = match.user_id
    job_hash = match.job_hash
    match_id = match.match_id

    user_data = users_dict.get(user_id, {})
    job_data  = jobs_dict.get(job_hash, {})
    clipboard = clips_dict.get(user_id, {})
    email_cfg = emails_dict.get(user_id, {})

    if not user_data or not job_data:
        return None, None, match_id, 'error'

    # AI tailor
    ai_content = ai_tailor_resume(user_data, job_data, clipboard)

    # Build unique filename
    resume_id   = str(uuid.uuid4())
    safe_name   = re.sub(r'[^a-zA-Z0-9]', '_', user_data.get('full_name', 'User'))[:20]
    safe_job    = re.sub(r'[^a-zA-Z0-9]', '_', job_data.get('job_title', 'Job'))[:20]
    file_stem   = f'{safe_name}_{safe_job}_{resume_id[:8]}'
    docx_local  = os.path.join(LOCAL_RESUME_DIR, f'{file_stem}.docx')
    docx_dbfs   = f'{DBFS_RESUME_DIR}{user_id}/{file_stem}.docx'
    pdf_dbfs    = f'{DBFS_RESUME_DIR}{user_id}/{file_stem}.pdf'

    # Build DOCX
    try:
        build_resume_docx(user_data, job_data, ai_content, clipboard, docx_local)
    except Exception as e:
        print(f'  ❌ DOCX build failed for {user_id}: {e}')
        return None, None, match_id, 'error'

    # Upload DOCX to DBFS
    try:
        with open(docx_local, 'rb') as f:
            raw = f.read()
        dbutils.fs.put(docx_dbfs, base64.b64encode(raw).decode(), overwrite=True)
    except Exception as e:
        print(f'  ⚠️  DBFS upload failed: {e}')

    skills_highlighted = ', '.join(ai_content.get('skills_highlighted', []))
    familiar           = ai_content.get('familiar_with', '')

    gen_record = {
        'resume_id':             resume_id,
        'user_id':               user_id,
        'job_hash':              job_hash,
        'match_id':              match_id,
        'clipboard_id':          clipboard.get('clipboard_id'),
        'docx_path':             docx_dbfs,
        'pdf_path':              pdf_dbfs,  # To be generated when user downloads
        'experience_years_used': float(clipboard.get('years_experience_override') or user_data.get('years_experience') or 0),
        'skills_highlighted':    skills_highlighted,
        'basic_knowledge_added': familiar,
        'tailored_bullets_json': json.dumps(ai_content.get('role_bullets', [])),
        'linkedin_used':         user_data.get('linkedin_url', ''),
        'resume_version':        1,
        'is_latest':             True,
        'generation_model':      NVIDIA_MODEL,
        'generated_at':          datetime.now(),
        'last_edited_at':        None,
        'is_user_edited':        False,
    }

    # Build email draft
    email_draft = build_email_draft(user_data, job_data, resume_id, match_id, email_cfg)

    return gen_record, email_draft, match_id, 'resume_ready'


# Run in parallel
with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(process_one_match, m): m for m in pending_matches}
    for future in as_completed(futures):
        try:
            gen_rec, email_draft, match_id, status = future.result(timeout=120)
            if gen_rec:
                generated_records.append(gen_rec)
                email_draft_records.append(email_draft)
                match_updates.append((match_id, gen_rec['resume_id'], status))
                print(f'  ✅ Resume created: {gen_rec["resume_id"][:8]}...')
        except Exception as e:
            print(f'  ❌ Future error: {e}')

print(f'\n🎉 Generated {len(generated_records)} resumes!')

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, TimestampType, DoubleType, DateType

# Save generated_resumes
if generated_records:
    GEN_SCHEMA = StructType([
        StructField('resume_id',             StringType(),    False),
        StructField('user_id',               StringType(),    True),
        StructField('job_hash',              StringType(),    True),
        StructField('match_id',              StringType(),    True),
        StructField('clipboard_id',          StringType(),    True),
        StructField('docx_path',             StringType(),    True),
        StructField('pdf_path',              StringType(),    True),
        StructField('experience_years_used', DoubleType(),    True),
        StructField('skills_highlighted',    StringType(),    True),
        StructField('basic_knowledge_added', StringType(),    True),
        StructField('tailored_bullets_json', StringType(),    True),
        StructField('linkedin_used',         StringType(),    True),
        StructField('resume_version',        IntegerType(),   True),
        StructField('is_latest',             BooleanType(),   True),
        StructField('generation_model',      StringType(),    True),
        StructField('generated_at',          TimestampType(), True),
        StructField('last_edited_at',        TimestampType(), True),
        StructField('is_user_edited',        BooleanType(),   True),
    ])
    gen_df = spark.createDataFrame(generated_records, schema=GEN_SCHEMA)
    gen_df.write.format('delta').mode('append').saveAsTable(GEN_RESUMES_TBL)
    print(f'✅ Saved {len(generated_records)} records to {GEN_RESUMES_TBL}')

# Save email drafts
if email_draft_records:
    EMAIL_SCHEMA = StructType([
        StructField('draft_id',           StringType(),    False),
        StructField('user_id',            StringType(),    True),
        StructField('job_hash',           StringType(),    True),
        StructField('match_id',           StringType(),    True),
        StructField('resume_id',          StringType(),    True),
        StructField('from_email',         StringType(),    True),
        StructField('to_email',           StringType(),    True),
        StructField('subject',            StringType(),    True),
        StructField('body_html',          StringType(),    True),
        StructField('body_text',          StringType(),    True),
        StructField('draft_type',         StringType(),    True),
        StructField('parent_draft_id',    StringType(),    True),
        StructField('recruiter_email_raw',StringType(),    True),
        StructField('status',             StringType(),    True),
        StructField('created_at',         TimestampType(), True),
        StructField('sent_at',            TimestampType(), True),
        StructField('smtp_response',      StringType(),    True),
        StructField('is_user_edited',     BooleanType(),   True),
    ])
    email_df = spark.createDataFrame(email_draft_records, schema=EMAIL_SCHEMA)
    email_df.write.format('delta').mode('append').saveAsTable(EMAIL_DRAFTS_TBL)
    print(f'✅ Saved {len(email_draft_records)} email drafts')

# Update match statuses
if match_updates:
    for match_id, resume_id, status in match_updates:
        spark.sql(f'''
            UPDATE {MATCHES_TABLE}
            SET resume_generated = true,
                resume_id = '{resume_id}',
                resume_generated_at = current_timestamp(),
                status = '{status}'
            WHERE match_id = '{match_id}'
        ''')
    print(f'✅ Updated {len(match_updates)} match records')

print(f'\n🎉 Resume generation pipeline complete!')